# QDIST v4.0.0 reviewed analytical preflight

This notebook standardizes the already-frozen QDIST v3.1.1 hard-clipping detector under the common G1-G10 and A-J framework. It does not overwrite the v3.1.1 freeze and does not change detector thresholds. The v4.0.0 family can proceed only if later cohort extraction proves exact numerical equivalence to the frozen v3.1.1 measurement.

QDIST measures native-waveform evidence compatible with hard clipping or saturation. It does not estimate total harmonic distortion, soft clipping, limiting, dynamic-range compression, automatic gain control, general codec distortion, or perceptual distortion.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'paper1_qc').exists():
            return candidate
    raise FileNotFoundError('Open this notebook from the paper_1 repository.')

ROOT = find_project_root()
for source in [ROOT / 'src', ROOT / 'src reviewed']:
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))

from paper1_qc_reviewed.qdist_v400 import (
    MEASUREMENT_VERSION, LEGACY_MEASUREMENT_VERSION, ANALYSIS_FEATURES,
    PREFLIGHT_HOTFIX_REVISION, PREFLIGHT_PANEL_STEMS,
    run_preflight, write_json,
)

OUTPUT_ROOT = ROOT / 'outputs reviewed' / 'nonlinear_distortion' / 'qdist-v4.0.0-candidate'
RUN_PACKAGE_TESTS = True
RUN_CODEC_CHARACTERIZATION = True
RUN_COHORT_EXTRACTION = False
PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = "PENDING"

DECLARED_PREFLIGHT_PANELS = (
    "A_construct_response",
    "B_discriminant_specificity",
    "C_transformation_contract",
)
assert DECLARED_PREFLIGHT_PANELS == PREFLIGHT_PANEL_STEMS

print('Project:', ROOT)
print('Reviewed measurement:', MEASUREMENT_VERSION)
print('Frozen numerical baseline:', LEGACY_MEASUREMENT_VERSION)
print('Analysis features:', ANALYSIS_FEATURES)
print('Preflight hotfix:', PREFLIGHT_HOTFIX_REVISION)
print('Declared preflight panels:', DECLARED_PREFLIGHT_PANELS)
print('Output root:', OUTPUT_ROOT)

Project: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1
Reviewed measurement: qdist-v4.0.0-candidate
Frozen numerical baseline: qdist-v3.1.1
Analysis features: ('qdist_hard_clipped_frame_fraction', 'qdist_hard_clip_event_rate_per_min', 'qdist_hard_clipped_sample_fraction')
Preflight hotfix: legacy-api-compat-r1
Declared preflight panels: ('A_construct_response', 'B_discriminant_specificity', 'C_transformation_contract')
Output root: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1\outputs reviewed\nonlinear_distortion\qdist-v4.0.0-candidate


In [2]:
package_tests_passed = False
package_test_output = ''
if RUN_PACKAGE_TESTS:
    command = [
        str(ROOT / '.venv' / 'Scripts' / 'python.exe'), '-m', 'pytest',
        str(ROOT / 'tests reviewed' / 'test_qdist_v400.py'),
        str(ROOT / 'tests reviewed' / 'test_qdist_v400_notebook.py'), '-q',
    ]
    completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    package_test_output = completed.stdout + completed.stderr
    print(package_test_output)
    package_tests_passed = completed.returncode == 0
    if not package_tests_passed:
        raise RuntimeError('Reviewed QDIST package tests failed.')
else:
    print('Package tests skipped by control.')

............................                                             [100%]



In [3]:
evidence = run_preflight(OUTPUT_ROOT, run_codecs=RUN_CODEC_CHARACTERIZATION)
checks = evidence['checks']
figure_index = evidence['figure_index']
manifest = evidence['manifest']
checks

,gate,check,passed,observed,required
0,G1,frozen production detector API complete,True,[],none missing
1,G1,legacy frozen measurement identity exact,True,qdist-v3.1.1,qdist-v3.1.1
2,G1,exact three-feature registry,True,"('qdist_hard_clipped_frame_fraction', 'qdist_h...","('qdist_hard_clipped_frame_fraction', 'qdist_h..."
3,G2,clean speech valid zero,True,"{'qdist_hard_clipped_frame_fraction': 0.0, 'qd...",zero accepted episodes
4,G2,hard clipping detected,True,"{'qdist_hard_clipped_frame_fraction': 0.17, 'q...",positive
5,G2,all three views reconstruct from ledgers,True,[{'feature': 'qdist_hard_clipped_frame_fractio...,absolute error <=1e-12
6,G3,polarity equivariance,True,"{'condition': 'polarity_inversion', 'qdist_har...",all features exact
7,G3,common time-shift invariance,True,"{'condition': 'common_time_shift', 'qdist_hard...",all features exact
8,G3,lossless PCM16 roundtrip,True,"{'condition': 'lossless_pcm16_roundtrip', 'qdi...",all features exact
9,G3,resampling and lossy codecs characterized,True,"[aac_96k, baseline, common_time_shift, lossles...",native plus characterization rows


In [4]:
failed = checks.loc[~checks['passed'].astype(bool)]
print('Blocking checks:', f"{int(checks['passed'].sum())}/{len(checks)}")
print('Figure panels:', sorted(figure_index['panel'].tolist()))
if len(failed):
    display(failed)
    raise RuntimeError('QDIST reviewed preflight has failed blocking checks.')
if RUN_COHORT_EXTRACTION:
    raise RuntimeError('The analytical preflight must not run cohort extraction.')

Blocking checks: 18/18
Figure panels: ['A', 'B', 'C']


In [5]:
manifest_path = OUTPUT_ROOT / 'manifests' / 'qdist_v400_preflight_candidate_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
manifest['package_tests_passed'] = bool(package_tests_passed)
manifest['candidate_only'] = True
manifest['cohort_extraction_completed'] = False
manifest['freeze_allowed'] = False
manifest['publish_and_freeze'] = False
manifest['scientific_review_decision'] = SCIENTIFIC_REVIEW_DECISION
manifest['family_scalar_constructed'] = False
manifest['standalone_gate_allowed'] = False
write_json(manifest_path, manifest)
manifest

{'analysis_features': ['qdist_hard_clipped_frame_fraction',
  'qdist_hard_clip_event_rate_per_min',
  'qdist_hard_clipped_sample_fraction'],
 'candidate_only': True,
 'cohort_extraction_completed': False,
 'complete_nonlinear_distortion_claim_allowed': False,
 'created_utc': '2026-08-04T14:25:05.040035+00:00',
 'family_scalar_constructed': False,
 'freeze_allowed': False,
 'legacy_contract': {'detector_entrypoint': 'extract_qdist',
  'detector_frame_length_ms': 30.0,
  'features_exact': True,
  'fixture_helpers_owned_by_reviewed_preflight': True,
  'legacy_features': ['qdist_hard_clipped_frame_fraction',
   'qdist_hard_clip_event_rate_per_min',
   'qdist_hard_clipped_sample_fraction'],
  'legacy_measurement_version': 'qdist-v3.1.1',
  'missing_symbols': [],
  'parameter_payload': {'absolute_flat_tolerance': 2e-07,
   'candidate_generation_minimum_edge_to_robust_peak_ratio': 0.25,
   'coarse_minimum_cluster_candidates': 2,
   'coarse_minimum_edge_to_interior_ratio': 3.0,
   'coarse_mini

In [6]:
assert manifest['preflight_hotfix_revision'] == 'legacy-api-compat-r1'
assert manifest['preflight_blocking_checks_pass']
assert manifest['package_tests_passed']
assert manifest['panels_complete'] == ['A', 'B', 'C']
assert manifest['panel_i_status'] == 'APPLICABLE_pending_event_verification'
assert not manifest['cohort_extraction_completed']
assert not manifest['freeze_allowed']
assert not manifest['family_scalar_constructed']
assert not manifest['standalone_gate_allowed']
print('QDIST v4.0.0 REVIEWED PREFLIGHT COMPLETE')
print('Candidate only. Cohort extraction, event verification, G7-G10 decisions, and freezing remain pending.')

QDIST v4.0.0 REVIEWED PREFLIGHT COMPLETE
Candidate only. Cohort extraction, event verification, G7-G10 decisions, and freezing remain pending.
